# Olist E-Commerce Operations & Customer Experience Analytics

## Business Analysis & Recommendations

### Business Objective

This analysis evaluates the operational performance and customer experience of the Olist e-commerce marketplace to identify areas where business and operational improvements could have the greatest impact.

The analysis focuses on four core business questions:

1. How significant is the late-delivery problem, and where in the delivery process are delays most strongly associated with late orders?
2. How is delivery performance associated with customer satisfaction?
3. Which regions, product categories, and sellers represent the most important operational risk areas?
4. What do payment behavior and repeat purchasing reveal about customer behavior?

The objective is to translate the exploratory analysis into actionable business insights and recommendations while distinguishing observed associations from causal conclusions.

## 1. How Significant Is the Late-Delivery Problem?

Late delivery is one of the clearest operational risks identified in the exploratory analysis.

Among delivered orders with measurable delivery performance:

- 96,470 delivered orders could be evaluated for lateness.
- 7,826 orders arrived after the estimated delivery date.
- The late-delivery rate was 8.11%.
- Median delivery time was 10.22 days.
- Late orders were associated with substantially longer carrier transit times than on-time orders.

The purpose of this section is to assess the operational importance of late delivery and identify which stage of the fulfillment process is most strongly associated with delayed orders.

In [1]:
import pandas as pd
import numpy as np

fact_orders = pd.read_csv(
    "../Data/processed/fact_orders.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

fact_orders["is_late"] = fact_orders["is_late"].astype("boolean")

In [2]:
delivered_orders = fact_orders[
    (fact_orders["order_status"] == "delivered")
    & fact_orders["is_late"].notna()
].copy()

delivery_kpis = pd.DataFrame({
    "Metric": [
        "Measurable Delivered Orders",
        "Late Orders",
        "Late Delivery Rate (%)",
        "Median Delivery Time (Days)",
        "Average Delivery Time (Days)"
    ],
    "Value": [
        len(delivered_orders),
        delivered_orders["is_late"].sum(),
        delivered_orders["is_late"].mean() * 100,
        delivered_orders["delivery_time_days"].median(),
        delivered_orders["delivery_time_days"].mean()
    ]
})

delivery_kpis.round(2)

,Metric,Value
0,Measurable Delivered Orders,96470.00
1,Late Orders,7826.00
2,Late Delivery Rate (%),8.11
3,Median Delivery Time (Days),10.22
4,Average Delivery Time (Days),12.56


In [3]:
delivery_stage_comparison = (
    delivered_orders
    .groupby("is_late")
    .agg(
        orders=("order_id", "count"),
        median_handoff_days=("carrier_handoff_days", "median"),
        median_transit_days=("carrier_transit_days", "median"),
        median_delivery_days=("delivery_time_days", "median")
    )
    .reset_index()
)

delivery_stage_comparison["delivery_status"] = (
    delivery_stage_comparison["is_late"]
    .map({
        False: "On-Time",
        True: "Late"
    })
)

delivery_stage_comparison[
    [
        "delivery_status",
        "orders",
        "median_handoff_days",
        "median_transit_days",
        "median_delivery_days"
    ]
].round(2)

,delivery_status,orders,median_handoff_days,median_transit_days,median_delivery_days
0,On-Time,88644,2.14,6.92,9.73
1,Late,7826,3.44,23.92,29.15


### Business Interpretation

Late delivery affects a meaningful minority of completed orders: 8.11% of measurable delivered orders arrived after their estimated delivery date.

The delivery-stage comparison shows that the largest difference between on-time and late orders occurs after carrier handoff:

- Median seller-to-carrier handoff time increased from 2.14 days for on-time orders to 3.44 days for late orders, a difference of 1.30 days.
- Median carrier transit time increased from 6.92 days to 23.92 days, a difference of 17.00 days.
- As a result, median total delivery time increased from 9.73 days for on-time orders to 29.15 days for late orders.

This indicates that extended transit time is the strongest observed operational signal associated with late delivery in this dataset. However, the analysis demonstrates association rather than proving that carriers are the direct cause of individual delays.

### Business Recommendation

Olist should prioritize monitoring the post-handoff delivery stage by tracking carrier transit performance across regions, sellers, and product categories.

Operational alerts could be created for shipments exceeding expected transit thresholds, allowing potentially delayed orders to be identified earlier. High-volume regions and marketplace partners with persistently elevated late-delivery rates should then be investigated separately to determine the underlying operational causes.

## 2. How Is Delivery Performance Associated with Customer Satisfaction?

Customer review scores provide a direct measure of the customer experience recorded in the marketplace.

This section compares review scores between on-time and late deliveries to determine whether delivery reliability is associated with customer satisfaction.

Only delivered orders with measurable late-delivery status and an available review score are included in the comparison.

In [4]:
review_delivery_analysis = delivered_orders[
    delivered_orders["latest_review_score"].notna()
].copy()

review_comparison = (
    review_delivery_analysis
    .groupby("is_late")
    .agg(
        reviewed_orders=("order_id", "count"),
        average_review_score=("latest_review_score", "mean"),
        median_review_score=("latest_review_score", "median")
    )
    .reset_index()
)

review_comparison["delivery_status"] = (
    review_comparison["is_late"]
    .map({
        False: "On-Time",
        True: "Late"
    })
)

review_comparison[
    [
        "delivery_status",
        "reviewed_orders",
        "average_review_score",
        "median_review_score"
    ]
].round(2)

,delivery_status,reviewed_orders,average_review_score,median_review_score
0,On-Time,88163,4.29,5.0
1,Late,7661,2.57,2.0


In [5]:
late_review_orders = review_delivery_analysis[
    review_delivery_analysis["is_late"] == True
].copy()

late_review_orders["delay_severity"] = pd.cut(
    late_review_orders["delivery_delay_days"],
    bins=[0, 3, 7, 14, 30, np.inf],
    labels=[
        "0–3 Days",
        "4–7 Days",
        "8–14 Days",
        "15–30 Days",
        "30+ Days"
    ],
    include_lowest=True
)

delay_review_summary = (
    late_review_orders
    .groupby("delay_severity", observed=True)
    .agg(
        reviewed_orders=("order_id", "count"),
        average_review_score=("latest_review_score", "mean"),
        median_review_score=("latest_review_score", "median")
    )
)

delay_review_summary.round(2)

,reviewed_orders,average_review_score,median_review_score
delay_severity,,,
0–3 Days,2636,3.77,4.0
4–7 Days,1773,2.32,1.0
8–14 Days,1748,1.74,1.0
15–30 Days,1161,1.61,1.0
30+ Days,343,2.03,1.0


### Business Interpretation

Delivery reliability is strongly associated with customer satisfaction.

On-time delivered orders received an average review score of 4.29, compared with only 2.57 for late orders, representing a 1.72-point difference on a five-point scale. The median review score also fell from 5 for on-time deliveries to 2 for late deliveries.

Delay severity provides additional evidence of this relationship:

- Orders delivered 0–3 days late received an average review score of 3.77.
- At 4–7 days late, the average declined to 2.32.
- At 8–14 days late, it declined further to 1.74.
- Orders delivered 15–30 days late had the lowest average score at 1.61.
- Orders more than 30 days late averaged 2.03, showing that the relationship is strong but not perfectly monotonic.

The results indicate that customer experience deteriorates substantially once delivery delays become more severe. However, these results demonstrate an association between delivery performance and review scores rather than proving that delivery delay is the only cause of lower customer satisfaction.

### Business Recommendation

Olist should treat late-delivery risk as both an operational and customer-experience issue.

Customers whose orders are predicted to miss the estimated delivery date could receive proactive delivery updates before the expected date passes. Orders experiencing increasingly severe delays should receive progressively higher intervention priority.

A practical customer-recovery process could prioritize:

1. Early notification when a delay becomes likely.
2. Updated delivery expectations for affected customers.
3. Escalation of orders experiencing extended delays.
4. Customer-service follow-up for severe delivery exceptions.

Reducing delivery uncertainty and responding earlier to delayed orders may help protect the customer experience even when operational delays cannot be completely avoided.

## 3. Which Regions Should Be Prioritized for Operational Improvement?

Late-delivery rate alone does not determine business priority.

A region with a high late-delivery rate but relatively few orders may affect fewer customers than a high-volume region with a moderately elevated late-delivery rate.

Therefore, regional prioritization should consider both:

- The percentage of delivered orders arriving late.
- The absolute number of late orders affecting customers.

This section evaluates state-level delivery performance using both measures to identify where operational improvements could have the greatest business impact.

In [6]:
state_performance = (
    delivered_orders
    .groupby("customer_state")
    .agg(
        delivered_orders=("order_id", "count"),
        late_orders=("is_late", "sum"),
        late_rate=("is_late", "mean"),
        median_delivery_days=("delivery_time_days", "median"),
        average_review_score=("latest_review_score", "mean")
    )
    .reset_index()
)

state_performance["late_rate_pct"] = (
    state_performance["late_rate"] * 100
)

state_priority = (
    state_performance[
        state_performance["delivered_orders"] >= 300
    ]
    .sort_values("late_orders", ascending=False)
)

state_priority[
    [
        "customer_state",
        "delivered_orders",
        "late_orders",
        "late_rate_pct",
        "median_delivery_days",
        "average_review_score"
    ]
].round(2)

,customer_state,delivered_orders,late_orders,late_rate_pct,median_delivery_days,average_review_score
25,SP,40494,2387,5.89,7.21,4.25
18,RJ,12350,1664,13.47,12.04,3.97
10,MG,11354,637,5.61,10.31,4.19
4,BA,3256,457,14.04,16.91,3.93
22,RS,5344,382,7.15,13.18,4.18
23,SC,3546,346,9.76,13.01,4.13
17,PR,4923,246,5.0,10.43,4.24
7,ES,1995,244,12.23,13.64,4.08
5,CE,1279,196,15.32,18.21,3.94
15,PE,1593,172,10.8,15.70,4.08


### Business Interpretation

Regional operational priority changes significantly when order volume is considered alongside late-delivery rate.

São Paulo (SP) had a relatively low late-delivery rate of 5.89%, but its very large order volume resulted in 2,387 late orders — the highest absolute number in the dataset.

Rio de Janeiro (RJ) represents a particularly important operational risk because it combines both high volume and weaker delivery performance. Among 12,350 measurable delivered orders, 1,664 arrived late, producing a late-delivery rate of 13.47%.

Bahia (BA) shows a similar pattern at a smaller scale, with a 14.04% late-delivery rate and 457 late orders.

By contrast, Alagoas (AL) recorded the highest late-delivery rate among states with at least 300 delivered orders at 23.93%, but this represented 95 late orders because overall order volume was much smaller.

These results demonstrate why operational prioritization should consider both failure rate and customer impact rather than ranking regions by late-delivery percentage alone.

### Business Recommendation

A two-level regional improvement strategy would provide a more useful operational framework:

**High-impact markets:**  
SP, RJ, and BA should receive attention because their order volumes result in large absolute numbers of affected customers. RJ is especially important because it combines high order volume with a substantially elevated late-delivery rate.

**High-risk markets:**  
States such as AL, MA, PI, CE, and SE should be investigated because their late-delivery rates are substantially elevated, even though their total order volumes are smaller.

Operational teams should track both late-delivery rate and absolute late-order volume in regional performance dashboards. This would help distinguish systemic high-rate problems from high-volume problems and allocate improvement resources accordingly.

## 4. Which Product Categories and Sellers Should Be Prioritized?

Operational performance can vary across product categories and marketplace sellers.

However, prioritization should not be based only on the highest late-delivery percentage. Order volume, delivery time, and customer review scores should also be considered to identify areas where operational problems may have greater business impact.

Because individual orders can contain products from multiple categories or multiple sellers, the results in this section represent outcomes associated with each category or seller rather than exclusive causal responsibility.

In [7]:
fact_order_items = pd.read_csv(
    "../Data/processed/fact_order_items.csv"
)

dim_products = pd.read_csv(
    "../Data/processed/dim_products.csv"
)

items_with_products = fact_order_items.merge(
    dim_products[
        ["product_id", "product_category_name_english"]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

order_category = (
    items_with_products[
        ["order_id", "product_category_name_english"]
    ]
    .drop_duplicates()
)

print("Order-category records:", len(order_category))
print("Unique orders:", order_category["order_id"].nunique())
print("Missing categories:", order_category["product_category_name_english"].isna().sum())

Order-category records: 99470
Unique orders: 98666
Missing categories: 0


In [8]:
category_order_performance = order_category.merge(
    fact_orders[
        [
            "order_id",
            "order_status",
            "is_late",
            "delivery_time_days",
            "latest_review_score"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)

category_delivery = category_order_performance[
    (category_order_performance["order_status"] == "delivered")
    & category_order_performance["is_late"].notna()
].copy()

category_performance = (
    category_delivery
    .groupby("product_category_name_english")
    .agg(
        delivered_orders=("order_id", "count"),
        late_orders=("is_late", "sum"),
        late_rate=("is_late", "mean"),
        median_delivery_days=("delivery_time_days", "median"),
        average_review_score=("latest_review_score", "mean")
    )
    .reset_index()
)

category_performance["late_rate_pct"] = (
    category_performance["late_rate"] * 100
)

category_priority = (
    category_performance[
        category_performance["delivered_orders"] >= 300
    ]
    .sort_values("late_orders", ascending=False)
)

category_priority[
    [
        "product_category_name_english",
        "delivered_orders",
        "late_orders",
        "late_rate_pct",
        "median_delivery_days",
        "average_review_score"
    ]
].head(15).round(2)

,product_category_name_english,delivered_orders,late_orders,late_rate_pct,median_delivery_days,average_review_score
10,bed_bath_table,9272,811,8.75,10.71,4.00
46,health_beauty,8647,775,8.96,9.55,4.23
68,sports_leisure,7529,584,7.76,10.11,4.23
42,furniture_decor,6307,535,8.48,10.87,4.06
18,computers_accessories,6529,503,7.7,11.10,4.08
73,watches_gifts,5493,468,8.52,10.34,4.12
52,housewares,5743,399,6.95,8.93,4.19
71,telephony,4093,349,8.53,10.81,4.05
8,auto,3809,328,8.61,9.74,4.15
72,toys,3803,286,7.52,9.41,4.24


In [9]:
category_high_risk = (
    category_performance[
        category_performance["delivered_orders"] >= 300
    ]
    .sort_values("late_rate_pct", ascending=False)
)

category_high_risk[
    [
        "product_category_name_english",
        "delivered_orders",
        "late_orders",
        "late_rate_pct",
        "median_delivery_days",
        "average_review_score"
    ]
].head(10).round(2)

,product_category_name_english,delivered_orders,late_orders,late_rate_pct,median_delivery_days,average_review_score
7,audio,348,45,12.93,10.78,3.84
50,home_confort,392,41,10.46,11.45,3.88
39,food,441,44,9.98,7.15,4.33
29,electronics,2517,247,9.81,10.99,4.12
9,baby,2809,258,9.18,10.05,4.10
60,office_furniture,1254,115,9.17,18.86,3.64
2,Unknown,1392,127,9.12,10.67,4.01
20,construction_tools_construction,736,67,9.1,8.70,4.13
46,health_beauty,8647,775,8.96,9.55,4.23
59,musical_instruments,611,54,8.84,10.38,4.23


### Product Category Interpretation

Product-category performance shows another important distinction between operational scale and operational risk.

By absolute late-order volume, the largest category-associated impacts were:

- Bed, bath & table: 811 late orders from 9,272 delivered orders.
- Health & beauty: 775 late orders from 8,647 delivered orders.
- Sports & leisure: 584 late orders from 7,529 delivered orders.
- Furniture & decor: 535 late orders from 6,307 delivered orders.
- Computers & accessories: 503 late orders from 6,529 delivered orders.

These categories do not necessarily have the highest late-delivery rates, but their large transaction volumes mean that delivery problems affect a greater number of orders.

Among categories with at least 300 delivered orders, audio recorded the highest late-delivery rate at 12.93%, although this represented only 45 late orders because category volume was relatively small.

Office furniture represents a particularly notable customer-experience risk. Its late-delivery rate was 9.17%, median delivery time reached 18.86 days, and its average review score was only 3.64 — one of the weakest combinations among the higher-volume categories analyzed.

### Business Recommendation

Category monitoring should distinguish between two types of operational priorities:

**High-impact categories** such as bed, bath & table and health & beauty should be monitored because even moderate late-delivery rates affect large numbers of customers.

**High-risk categories** such as audio and office furniture should receive targeted investigation because their delivery or customer-experience metrics are unusually weak relative to other categories.

Olist could incorporate category-level late-delivery rate, late-order volume, median delivery time, and review score into an operational scorecard to identify categories requiring deeper investigation.

Because orders may contain multiple product categories, these results represent category-associated outcomes and should not be interpreted as proof that the product category itself caused the delivery outcome.

In [10]:
order_seller = (
    fact_order_items[
        ["order_id", "seller_id"]
    ]
    .drop_duplicates()
)

seller_order_performance = order_seller.merge(
    fact_orders[
        [
            "order_id",
            "order_status",
            "is_late",
            "delivery_time_days",
            "latest_review_score"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("Order-seller records:", len(order_seller))
print("Unique orders:", order_seller["order_id"].nunique())
print("Unique sellers:", order_seller["seller_id"].nunique())

Order-seller records: 100010
Unique orders: 98666
Unique sellers: 3095


In [11]:
seller_delivery = seller_order_performance[
    (seller_order_performance["order_status"] == "delivered")
    & seller_order_performance["is_late"].notna()
].copy()

seller_performance = (
    seller_delivery
    .groupby("seller_id")
    .agg(
        delivered_orders=("order_id", "count"),
        late_orders=("is_late", "sum"),
        late_rate=("is_late", "mean"),
        median_delivery_days=("delivery_time_days", "median"),
        average_review_score=("latest_review_score", "mean")
    )
    .reset_index()
)

seller_performance["late_rate_pct"] = (
    seller_performance["late_rate"] * 100
)

seller_priority = (
    seller_performance[
        seller_performance["delivered_orders"] >= 100
    ]
    .sort_values("late_rate_pct", ascending=False)
)

seller_priority[
    [
        "seller_id",
        "delivered_orders",
        "late_orders",
        "late_rate_pct",
        "median_delivery_days",
        "average_review_score"
    ]
].head(15).round(2)

,seller_id,delivered_orders,late_orders,late_rate_pct,median_delivery_days,average_review_score
79,06a2c3af7b3aee5d69171b0e14f0ee87,389,90,23.14,15.72,4.01
323,1ca7077d890b907f89be8c954a02686a,108,24,22.22,13.60,2.39
1606,88460e8ebdecbfecb5f9601833981930,246,48,19.51,15.35,3.43
2657,e5a3438891c0bfdb9394643f95273d8e,216,40,18.52,13.04,3.99
2404,cd68562d3f44870c08922d380acae552,122,22,18.03,13.35,4.02
1532,8160255418d5aaa7dbdc9f4c64ebda44,380,63,16.58,13.28,3.95
513,2c9e548be18521d1c43cde1c582c6de8,124,20,16.13,9.07,4.13
2577,dd7ddc04e1b6c2c614352b383efe2d36,121,19,15.7,12.59,3.88
2877,f7ba60f8c3f99e7ee4042fdef03b70c4,218,34,15.6,10.32,4.22
781,431af27f296bc6519d890aa5a05fdb11,116,18,15.52,14.97,3.80


In [12]:
seller_high_impact = (
    seller_performance[
        seller_performance["delivered_orders"] >= 100
    ]
    .sort_values("late_orders", ascending=False)
)

seller_high_impact[
    [
        "seller_id",
        "delivered_orders",
        "late_orders",
        "late_rate_pct",
        "median_delivery_days",
        "average_review_score"
    ]
].head(15).round(2)

,seller_id,delivered_orders,late_orders,late_rate_pct,median_delivery_days,average_review_score
858,4a3ca9315b744ce9f8e9374361493884,1772,195,11.0,12.06,3.85
358,1f50f920176fa81dab994f9023523100,1399,148,10.58,14.02,4.14
834,4869f7a5dfa277a7dca6462dcf3b52b2,1124,130,11.57,12.11,4.15
2725,ea8482cd71df3c1969d7b9473ff13abc,1132,118,10.42,10.87,4.03
1190,6560211a19b47992c3666cc44a7e94c0,1819,117,6.43,7.81,3.98
2388,cc419e0650a3c5ba77189a1882b7556a,1651,101,6.12,9.30,4.15
2543,da8622b14eb17ae2831f4ac5b9dab84a,1311,100,7.63,8.20,4.18
1758,955fee9216a65b617aa5c0531780ce60,1261,98,7.77,8.32,4.20
1480,7c67e1448b00f6e969d365cea6b010ab,973,98,10.07,19.83,3.50
1644,8b321bb669392f5163d04c59e235e066,930,97,10.43,10.65,4.10


### Seller Performance Interpretation

Seller-associated performance also shows an important distinction between operational risk and operational impact.

Among sellers with at least 100 measurable delivered orders, seller `06a2c3af7b3aee5d69171b0e14f0ee87` recorded the highest late-delivery rate at 23.14%, with 90 late orders among 389 delivered orders. This is nearly three times the marketplace-wide late-delivery rate of 8.11%.

Seller `1ca7077d890b907f89be8c954a02686a` also represents a notable customer-experience risk, combining a 22.22% late-delivery rate with an average review score of only 2.39.

When absolute customer impact is considered, different sellers become important. Seller `4a3ca9315b744ce9f8e9374361493884` generated 195 late orders from 1,772 delivered orders, the largest late-order count among sellers in the filtered analysis, despite a lower late-delivery rate of 11.00%.

Seller `7c67e1448b00f6e969d365cea6b010ab` also stands out across several indicators, with 98 late orders, a 10.07% late-delivery rate, a median delivery time of 19.83 days, and an average review score of 3.50.

These results show that seller monitoring should consider multiple dimensions rather than relying on a single KPI.

### Business Recommendation

Olist should develop a seller operational scorecard combining:

- Delivered-order volume
- Number of late orders
- Late-delivery rate
- Median delivery time
- Customer review score

High-volume sellers generating large numbers of late orders should be prioritized for operational improvement because they affect more customers, while sellers with unusually high late-delivery rates or weak review scores should be flagged for targeted investigation.

Persistent underperformance could trigger deeper analysis of seller preparation time, product mix, destination geography, and other available operational factors before corrective actions are determined.

Because some marketplace orders contain products from multiple sellers, these metrics represent outcomes associated with each seller rather than proving that an individual seller caused the complete delivery outcome.

## 5. What Does Repeat Purchasing Reveal About Customer Behavior?

Repeat purchasing provides an indication of customer retention within the observed marketplace period.

For this analysis, customers are identified using `customer_unique_id`, which allows multiple order records belonging to the same customer to be connected.

Only delivered orders are counted as completed purchases. A repeat customer is defined as a customer with at least two delivered orders during the period covered by the dataset.

Because the dataset represents a limited observation period, these metrics describe repeat-purchase behavior within the available data rather than a complete lifetime customer-retention rate.

In [13]:
customer_frequency = (
    fact_orders[
        fact_orders["order_status"] == "delivered"
    ]
    .groupby("customer_unique_id")
    .agg(
        delivered_orders=("order_id", "count")
    )
    .reset_index()
)

customer_frequency["is_repeat_customer"] = (
    customer_frequency["delivered_orders"] >= 2
)

total_customers = len(customer_frequency)
repeat_customers = customer_frequency["is_repeat_customer"].sum()

total_delivered_orders = customer_frequency["delivered_orders"].sum()

repeat_customer_orders = customer_frequency.loc[
    customer_frequency["is_repeat_customer"],
    "delivered_orders"
].sum()

print("Customers with delivered orders:", total_customers)
print("Repeat customers:", repeat_customers)
print(
    "Repeat customer rate:",
    round(repeat_customers / total_customers * 100, 2),
    "%"
)
print("Delivered orders:", total_delivered_orders)
print("Orders from repeat customers:", repeat_customer_orders)
print(
    "Share of delivered orders from repeat customers:",
    round(repeat_customer_orders / total_delivered_orders * 100, 2),
    "%"
)

Customers with delivered orders: 93358
Repeat customers: 2801
Repeat customer rate: 3.0 %
Delivered orders: 96478
Orders from repeat customers: 5921
Share of delivered orders from repeat customers: 6.14 %


### Business Interpretation

Repeat purchasing was relatively limited during the period observed in the dataset.

Among 93,358 customers with at least one delivered order, 2,801 customers completed two or more delivered purchases. This represents a repeat-customer rate of 3.00%.

Repeat customers generated 5,921 delivered orders, representing 6.14% of all 96,478 delivered orders.

The results indicate that most customers in the observed dataset completed only one purchase, while a relatively small customer segment returned for additional transactions.

However, this should not be interpreted as a complete lifetime customer-retention rate. The dataset covers a limited observation period, and customers who first purchased near the end of the dataset had less opportunity to make another purchase.

### Business Recommendation

Olist should investigate opportunities to increase repeat purchasing among customers who successfully complete their first transaction.

Potential retention initiatives could include:

- Post-purchase re-engagement campaigns.
- Personalized product recommendations based on previous purchases.
- Incentives for a second purchase.
- Customer segmentation based on purchase frequency and transaction behavior.
- Targeted retention strategies for customers with strong previous delivery and review experiences.

Future analysis should also examine repeat purchasing using customer cohorts, where customers are grouped by the month of their first purchase and given comparable time windows to make a subsequent purchase.

This would provide a more robust view of customer retention than the overall repeat-purchase percentage alone.

## 6. Executive Business Summary & Priority Action Plan

The analysis identifies delivery reliability as the strongest operational issue associated with customer experience in the Olist marketplace.

### Key Business Findings

**1. Late delivery represents a meaningful operational problem.**  
Among 96,470 measurable delivered orders, 7,826 arrived after the estimated delivery date, producing an overall late-delivery rate of 8.11%.

The largest observed difference between on-time and late orders occurred during the post-handoff transit stage. Median carrier transit time increased from 6.92 days for on-time orders to 23.92 days for late orders.

**2. Delivery reliability is strongly associated with customer satisfaction.**  
On-time orders received an average review score of 4.29, compared with only 2.57 for late orders.

Customer satisfaction also generally deteriorated as delays became more severe, indicating that extended delivery delays represent both an operational and customer-experience risk.

**3. Regional priority depends on both failure rate and business scale.**  
São Paulo generated the largest absolute number of late orders, with 2,387, despite a relatively low late-delivery rate of 5.89%.

Rio de Janeiro represents a particularly important improvement opportunity because it combined high order volume with a 13.47% late-delivery rate, resulting in 1,664 late orders.

Smaller markets including Alagoas and Maranhão showed substantially higher late-delivery rates and should be monitored as high-risk regions.

**4. Product categories and sellers show concentrated operational risks.**  
High-volume categories such as bed, bath & table and health & beauty generated the largest numbers of category-associated late orders.

Office furniture showed a particularly weak combination of operational and customer-experience metrics, including a 9.17% late-delivery rate, 18.86-day median delivery time, and 3.64 average review score.

Seller performance also varied substantially. Some sellers generated large absolute numbers of late orders, while others recorded late-delivery rates exceeding 20%.

**5. Repeat purchasing was limited within the observed dataset period.**  
Only 3.00% of customers with delivered orders completed at least two delivered purchases. Repeat customers generated 6.14% of delivered orders.

This suggests an opportunity to investigate customer-retention strategies, although cohort-based analysis would be required before drawing stronger conclusions about long-term retention.

### Priority Action Plan

Based on the analysis, the recommended business priorities are:

1. **Improve post-handoff delivery monitoring** by tracking carrier transit time and creating alerts for shipments showing elevated delay risk.
2. **Prioritize high-impact regions**, particularly Rio de Janeiro, while separately investigating smaller states with unusually high late-delivery rates.
3. **Implement proactive customer communication** when orders are likely to miss their estimated delivery dates.
4. **Create seller and category operational scorecards** combining volume, late-order count, late-delivery rate, delivery time, and customer review performance.
5. **Develop customer-retention analysis and campaigns** focused on encouraging a second purchase after a successful first transaction.

Overall, the analysis suggests that improving delivery reliability — particularly identifying extended transit delays earlier — represents one of the clearest opportunities to improve both marketplace operations and customer experience.

## 7. Analysis Limitations

The findings in this project should be interpreted within several important limitations:

- **Association does not imply causation.** Lower review scores are strongly associated with late deliveries, but the available data does not prove that delivery delay was the only cause of customer dissatisfaction.
- **Multi-seller and multi-category orders exist.** Delivery outcomes associated with a seller or product category cannot always be attributed exclusively to that seller or category.
- **Carrier-level attribution is limited.** The available dataset does not identify individual logistics carriers, preventing direct comparison of carrier performance.
- **Repeat purchasing is observation-period dependent.** Customers entering the dataset near the end of the available period had less time to make another purchase, so the 3.00% repeat-customer rate should not be interpreted as a lifetime retention rate.
- **Historical coverage is uneven at the dataset boundaries.** The earliest and latest months contain relatively few orders, so they should not be interpreted as normal full-month marketplace activity.
- **Some operational records contain missing or inconsistent timestamps.** These records were preserved during data cleaning but excluded from calculations where the required timestamps were unavailable or invalid.
- **Missing reviews represent unavailable feedback rather than negative feedback.** Orders without review data were not assigned artificial review scores.

These limitations were considered throughout the analysis to avoid overstating conclusions and to maintain a clear distinction between observed patterns and causal explanations.

In [14]:
fact_orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_time_days',
 'delivery_delay_days',
 'is_late',
 'carrier_handoff_days',
 'invalid_carrier_handoff',
 'carrier_transit_days',
 'invalid_carrier_transit',
 'approval_time_hours',
 'estimated_delivery_days',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'total_item_price',
 'total_freight_value',
 'total_item_transaction_value',
 'item_count',
 'total_payment_value',
 'payment_record_count',
 'payment_method_count',
 'max_installments',
 'latest_review_score',
 'review_record_count',
 'average_review_score',
 'min_review_score',
 'max_review_score',
 'has_review_comment']

In [15]:
fact_order_items.columns.tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'item_total_value',
 'freight_to_price_ratio']

In [17]:
fact_payments = pd.read_csv("../Data/processed/fact_payments.csv")

In [18]:
fact_payments.columns.tolist()

['order_id',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value']

In [19]:
fact_reviews = pd.read_csv("../Data/processed/fact_reviews.csv")

In [20]:
fact_reviews.columns.tolist()

['review_id',
 'order_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp',
 'has_review_title',
 'has_review_comment',
 'review_response_time_hours']

In [21]:
dim_customers = pd.read_csv("../Data/processed/dim_customers.csv")

In [22]:
dim_customers.columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [23]:
dim_products = pd.read_csv("../Data/processed/dim_products.csv")

In [24]:
dim_products.columns.tolist()

['product_id',
 'product_category_name',
 'product_name_length',
 'product_description_length',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'product_category_name_english']

In [25]:
dim_sellers = pd.read_csv("../Data/processed/dim_sellers.csv")

In [26]:
dim_sellers.columns.tolist()

['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

In [27]:
dim_geolocation_zip = pd.read_csv("../Data/processed/dim_geolocation_zip.csv")

In [28]:
dim_geolocation_zip.columns.tolist()

['geolocation_zip_code_prefix',
 'geolocation_lat',
 'geolocation_lng',
 'geolocation_city',
 'geolocation_state']